In [1]:
!pip install cassandra-driver -q

In [2]:
# Importar librerías necesarias
from cassandra.cluster import Cluster
from cassandra.query import BatchStatement, SimpleStatement

import uuid
from datetime import datetime, timedelta
import random

### Instrucciones para crear el contenedor de Cassandra

Para ejecutar Cassandra en un contenedor Docker, sigue estos pasos:

1. Asegúrate de tener Docker instalado y en ejecución en tu máquina.

2. Ejecuta el siguiente comando en tu terminal para descargar la imagen oficial de Cassandra y crear un contenedor:

   ```bash
   docker run --name lab_cassandra -d -p 9042:9042 cassandra:latest


In [3]:
# Configuración inicial
# Conexión al contenedor de Cassandra
# IMPORTANTE: El contenedor de Cassanda es lento, debes esperar a que esté operativo
cluster = Cluster(['127.0.0.1'])  # Asegúrate de que tu contenedor esté corriendo
session = cluster.connect()

In [4]:
# Visualizar las bases de datos existentes
def list_keyspaces():
    rows = session.execute("SELECT keyspace_name FROM system_schema.keyspaces;")
    print("Keyspaces existentes:")
    for row in rows:
        print("-", row.keyspace_name)

In [5]:
list_keyspaces()

Keyspaces existentes:
- biblioteca_space
- system_auth
- system_schema
- system_distributed
- system
- system_traces


In [6]:
# Crear un keyspace llamado Lab_UNI_Cass01
session.execute("""
CREATE KEYSPACE IF NOT EXISTS lab_uni_cass01 
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};
""")

In [7]:
list_keyspaces()

Keyspaces existentes:
- lab_uni_cass01
- biblioteca_space
- system_auth
- system_schema
- system_distributed
- system
- system_traces


In [8]:
# Usar el keyspace creado
session.set_keyspace('lab_uni_cass01')# Usar el keyspace creado

In [9]:
# Crear un esquema denormalizado para apuestas
# Tablas sin UDT para evitar errores
session.execute("""
CREATE TABLE IF NOT EXISTS bets (
    bet_id UUID PRIMARY KEY,
    user_id UUID,
    betting_house TEXT,
    bet_type TEXT,
    timestamp TIMESTAMP,
    online_game TEXT,
    online_bet_amount DECIMAL,
    online_win_amount DECIMAL,
    sports_sport TEXT,
    sports_team TEXT,
    sports_odds DECIMAL,
    sports_bet_amount DECIMAL,
    international_country TEXT,
    international_event TEXT,
    international_bet_amount DECIMAL,
    international_win_amount DECIMAL
);
""")

In [10]:
session.execute("""
CREATE TABLE IF NOT EXISTS casino_bets (
    casino_bet_id UUID PRIMARY KEY,
    user_id UUID,
    betting_house TEXT,
    game TEXT,
    bet_amount DECIMAL,
    win_amount DECIMAL,
    timestamp TIMESTAMP
) WITH default_time_to_live = 86400;  -- Datos expiran después de 1 día
""")

In [11]:
# Crear índices normales
session.execute("""
CREATE INDEX ON bets (online_game);
""")

In [12]:
session.execute("""
CREATE INDEX ON bets (bet_type);
""")

In [13]:
# Operaciones CRUD (Create, Read, Update, Delete)
def create_bet():
    bet_id = uuid.uuid4()
    user_id = uuid.uuid4()
    session.execute(
        """
        INSERT INTO bets (bet_id, user_id, betting_house, bet_type, timestamp, online_game, online_bet_amount, online_win_amount)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """,
        (
            bet_id,
            user_id,
            'HouseA',
            'online',
            datetime.now(),
            'Poker',
            50.0,
            100.0
        )
    )
    print("Bet created:", bet_id)

In [14]:
create_bet()

Bet created: 9646623f-9df4-4417-9cb9-fe065d9b19dc


In [15]:
def read_bets():
    rows = session.execute("SELECT * FROM bets;")
    for row in rows:
        print(row)

In [16]:
read_bets()

Row(bet_id=UUID('9646623f-9df4-4417-9cb9-fe065d9b19dc'), bet_type='online', betting_house='HouseA', international_bet_amount=None, international_country=None, international_event=None, international_win_amount=None, online_bet_amount=Decimal('50.0'), online_game='Poker', online_win_amount=Decimal('100.0'), sports_bet_amount=None, sports_odds=None, sports_sport=None, sports_team=None, timestamp=datetime.datetime(2026, 5, 25, 20, 50, 27, 993000), user_id=UUID('7093b4e0-ed21-48f2-845c-7a521d804351'))


In [17]:
def update_bet(bet_id):
    session.execute(
        """
        UPDATE bets SET online_bet_amount = %s WHERE bet_id = %s
        """,
        (75.0, bet_id)
    )
    print("Bet updated")

In [18]:
# Aquí puedes actualizar un bet_id generado previamente
# update_bet(<bet_id>)

def delete_bet(bet_id):
    session.execute(
        """
        DELETE FROM bets WHERE bet_id = %s
        """,
        (bet_id,)
    )
    print("Bet deleted")

# delete_bet(<bet_id>)

In [19]:
# Ejemplo de BATCH
batch = BatchStatement()
batch.add(
    SimpleStatement(
        """
        INSERT INTO bets (bet_id, user_id, betting_house, bet_type, timestamp, online_game, online_bet_amount, online_win_amount)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
    ),
    (
        uuid.uuid4(),
        uuid.uuid4(),
        'HouseB',
        'online',
        datetime.now(),
        'Roulette',
        100.0,
        200.0
    )
)

<BatchStatement type=LOGGED, statements=1, consistency=Not Set>

In [20]:
batch.add(
    SimpleStatement(
        """
        INSERT INTO casino_bets (casino_bet_id, user_id, betting_house, game, bet_amount, win_amount, timestamp)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """
    ),
    (
        uuid.uuid4(),
        uuid.uuid4(),
        'HouseB',
        'Blackjack',
        150.0,
        300.0,
        datetime.now()
    )
)

<BatchStatement type=LOGGED, statements=2, consistency=Not Set>

In [21]:
session.execute(batch)
print("Batch executed.")

Batch executed.


In [22]:
# Generación de datos simulados para casas de apuestas
betting_houses = ['HouseA', 'HouseB', 'HouseC', 'HouseD']
games = ['Poker', 'Blackjack', 'Roulette', 'Slots']
sports = ['Football', 'Basketball', 'Tennis', 'Horse Racing']

In [23]:
def generate_data():
    for _ in range(50):
        house = random.choice(betting_houses)
        game = random.choice(games)
        sport = random.choice(sports)
        bet_type = random.choice(['online', 'sports', 'international'])

        if bet_type == 'online':
            session.execute(
                """
                INSERT INTO bets (bet_id, user_id, betting_house, bet_type, timestamp, online_game, online_bet_amount, online_win_amount)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    uuid.uuid4(),
                    uuid.uuid4(),
                    house,
                    'online',
                    datetime.now(),
                    game,
                    random.uniform(10, 500),
                    random.uniform(0, 1000)
                )
            )
        elif bet_type == 'sports':
            session.execute(
                """
                INSERT INTO bets (bet_id, user_id, betting_house, bet_type, timestamp, sports_sport, sports_team, sports_odds, sports_bet_amount)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    uuid.uuid4(),
                    uuid.uuid4(),
                    house,
                    'sports',
                    datetime.now(),
                    sport,
                    "TeamA",
                    random.uniform(1.5, 3.5),
                    random.uniform(10, 500)
                )
            )
        elif bet_type == 'international':
            session.execute(
                """
                INSERT INTO bets (bet_id, user_id, betting_house, bet_type, timestamp, international_country, international_event, international_bet_amount, international_win_amount)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    uuid.uuid4(),
                    uuid.uuid4(),
                    house,
                    'international',
                    datetime.now(),
                    "CountryA",
                    sport,
                    random.uniform(10, 500),
                    random.uniform(0, 1000)
                )
            )


In [24]:
generate_data()
print("Data simulation completed.")

Data simulation completed.


In [25]:
# Crear índices normales
session.execute("""
CREATE TYPE telefono (
    cod_pais int,
    numero text,
);
""")

In [26]:
session.execute("""
CREATE TYPE direccion (
    calle text,
    ciudad text,
    poblacion text,
    pais text,
    cp text,
    telefonos map<text, frozen<telefono>>
);
""")

In [27]:
session.execute("""
CREATE TABLE usuario (
    nombre text PRIMARY KEY,
    direcciones map<text, frozen<direccion>>
);
""")

In [28]:
session.execute("""
INSERT INTO usuario (nombre, direcciones)
VALUES ('Pepe Armando', {
    'casa': {
        calle: 'General Ricardo',
        ciudad: 'Madrid',
        poblacion: 'Madrid',
        pais: 'Espana',
        cp: '28050',
        telefonos: {
            'movil_personal': {cod_pais: 35, numero: '78541259'},
            'movil_oficina': {cod_pais: 34, numero: '666888963'}
        }
    },
    'oficina': {
        calle: 'Plaza Castilla',
        ciudad: 'Madrid',
        poblacion: 'Centro',
        pais: 'Madrid',
        cp: '28036',
        telefonos: {
            'fax': {cod_pais: 33, numero: 'X'}
        }
    }
}
);
""")

In [29]:
session.execute("""
ALTER TYPE direccion RENAME cp TO CodPostal;
""")

#### Crear una Tabla para Almacenar Datos en Formato JSON

In [30]:
# Crear tabla
session.execute("""
CREATE TABLE IF NOT EXISTS users_point (
    id UUID PRIMARY KEY,
    data TEXT
)
""")

#### Insertar Datos en Formato JSON

In [31]:
import json

# Insertar datos JSON
user_data = {
    "name": "Alice",
    "age": 30,
    "email": "alice@example.com"
}

# Convertir el diccionario a una cadena JSON
user_data_json = json.dumps(user_data)

# Insertar los datos en la tabla
session.execute("""
INSERT INTO users_point (id, data) VALUES (%s, %s)
""", (uuid.uuid4(), user_data_json))

#### Consultar Datos como JSON

In [32]:
rows = session.execute("SELECT JSON * FROM users_point")

for row in rows:
    print(row.json)  # Resultado en formato JSON

{"id": "6bf3923b-596b-485b-8830-2df4013b242a", "data": "{\"name\": \"Alice\", \"age\": 30, \"email\": \"alice@example.com\"}"}


In [33]:
rows = session.execute("SELECT * FROM users_point")

for row in rows:
    # Convertir la cadena JSON a un diccionario de Python
    user_data = json.loads(row.data)
    print(f"ID: {row.id}, Nombre: {user_data['name']}, Edad: {user_data['age']}, Email: {user_data['email']}")

ID: 6bf3923b-596b-485b-8830-2df4013b242a, Nombre: Alice, Edad: 30, Email: alice@example.com


#### Actualizar Datos en Formato JSON

In [34]:
# Supongamos que conocemos el ID del usuario a actualizar
user_id = uuid.UUID('7e0e63cd-ae2f-4406-a344-879d2cd6459f')

# Nuevos datos del usuario
updated_user_data = {
    "name": "Alice Smith",
    "age": 31,
    "email": "alice.smith@example.com"
}

# Convertir el diccionario a una cadena JSON
updated_user_data_json = json.dumps(updated_user_data)

# Actualizar los datos en la tabla
session.execute("""
UPDATE users_point SET data = %s WHERE id = %s
""", (updated_user_data_json, user_id))

#### Convertir Filas a Objetos JSON

In [35]:
import json

rows = session.execute("SELECT * FROM users_point")

for row in rows:
    # Convertir a diccionario Python
    user_dict = {"id": str(row.id), "data": json.loads(row.data)}
    print(json.dumps(user_dict, indent=4))  # Formatear como JSON

{
    "id": "6bf3923b-596b-485b-8830-2df4013b242a",
    "data": {
        "name": "Alice",
        "age": 30,
        "email": "alice@example.com"
    }
}
{
    "id": "7e0e63cd-ae2f-4406-a344-879d2cd6459f",
    "data": {
        "name": "Alice Smith",
        "age": 31,
        "email": "alice.smith@example.com"
    }
}


#### Borrar Datos

In [36]:
# Borrar un registro por ID
session.execute("DELETE FROM users_point WHERE id = %s", (user_id,))

#### PROBLEMA: Consultas Avanzadas Usando JSON
`No es posible acceder a las claves del JSON`

In [37]:
# Consultar usuarios mayores de 25 años (filtrando dentro del JSON)
rows = session.execute("""
SELECT * FROM users_point WHERE data->'age' >= 25 ALLOW FILTERING
""")

for row in rows:
    print(row)

SyntaxException: <Error from server: code=2000 [Syntax error in CQL query] message="line 2:36 no viable alternative at input '-' (...SELECT * FROM users_point WHERE [data]-...)">

Nota: Usa ALLOW FILTERING con precaución, ya que puede impactar en el rendimiento.

#### Usar INSERT JSON para Insertar Datos

- Cassandra permite insertar datos JSON directamente en tablas con columnas definidas.
- El JSON debe tener claves que coincidan exactamente con los nombres de las columnas en la tabla.

In [ ]:
session.execute("""
CREATE TABLE IF NOT EXISTS users_json (
    id UUID PRIMARY KEY,
    name TEXT,
    age INT,
    email TEXT
)
""")

In [ ]:
user_data = {
    "id": str(uuid.uuid4()),
    "name": "Alice",
    "age": 30,
    "email": "alice@example.com"
}

session.execute("""
INSERT INTO users_json JSON %s
""", [json.dumps(user_data)])

#### Consultar Datos en Formato JSON
- Puedes recuperar datos en formato JSON con SELECT JSON.

In [ ]:
rows = session.execute("SELECT JSON * FROM users_json")

for row in rows:
    print(row.json)  # Devuelve una cadena JSON

#### Actualizar Datos
- Cassandra no permite actualizar directamente una fila utilizando JSON completo.
- Debes realizar una actualización estándar usando CQL:

In [ ]:
# Supongamos que conoces el ID del usuario
user_id = uuid.UUID('coloca-tu-id-aquí')

session.execute("""
UPDATE users_json SET name = %s, age = %s, email = %s WHERE id = %s
""", ("Alice Smith", 31, "alice.smith@example.com", user_id))

### Alternativa: Desnormalizar los Datos

- Si necesitas hacer consultas estructuradas sobre datos JSON, considera desnormalizar tus datos y dividir el contenido del JSON en múltiples columnas en lugar de mantenerlos como una única columna de texto.

##### Rediseñar la Tabla

In [ ]:
session.execute("""
CREATE TABLE IF NOT EXISTS users_structured (
    id UUID PRIMARY KEY,
    name TEXT,
    age INT,
    email TEXT
)
""")

##### Insertar Datos
Puedes insertar datos en la tabla utilizando una consulta INSERT INTO.

In [ ]:
import uuid

# Insertar un usuario
session.execute("""
INSERT INTO users_structured (id, name, age, email)
VALUES (%s, %s, %s, %s)
""", (uuid.uuid4(), "Alice", 30, "alice@example.com"))

# Insertar otro usuario
session.execute("""
INSERT INTO users_structured (id, name, age, email)
VALUES (%s, %s, %s, %s)
""", (uuid.uuid4(), "Bob", 25, "bob@example.com"))

##### Insertar Múltiples Usuarios
Para insertar varios usuarios en un solo paso, puedes usar un bucle:

In [ ]:
users = [
    {"name": "Charlie", "age": 35, "email": "charlie@example.com"},
    {"name": "Diana", "age": 28, "email": "diana@example.com"},
    {"name": "Eve", "age": 40, "email": "eve@example.com"}
]

for user in users:
    session.execute("""
    INSERT INTO users_structured (id, name, age, email)
    VALUES (%s, %s, %s, %s)
    """, (uuid.uuid4(), user["name"], user["age"], user["email"]))

##### Consultas Avanzadas
Puedes hacer consultas basadas en condiciones específicas:

In [ ]:
rows = session.execute("SELECT * FROM users_structured WHERE age > 25 ALLOW FILTERING")

for row in rows:
    print(f"Name: {row.name}, Age: {row.age}, Email: {row.email}")

#### Cerrar la Conexión

In [ ]:
cluster.shutdown()